# Cross-Staff Calibration Campaign 001

**Notebook:** 03 Geometry Model  
**Survey:** SUR-CAL-2026-001  
**Purpose:** Compare the observed staff readings with the geometric relationship expected for a cross-staff.  
**Author:** Dennis Hazelett

## 1. Introduction

This notebook introduces the first physically motivated model for the cross-staff calibration survey.

The analysis distinguishes among:

- known calibration geometry,
- raw instrument readings,
- quantities derived from a geometric model.

No canonical observation is modified. Calculated angles, predicted readings, residuals, and fitted parameters are analysis products.

## 2. Geometric Model

For a target of physical width $W$ observed from distance $D$, the target's angular width is

$$\theta = 2\arctan\left(\frac{W}{2D}\right).$$

For an ideal cross-staff with eye-to-crosspiece distance $L$ and crosspiece width $F$, the same angular width is

$$\theta = 2\arctan\left(\frac{F}{2L}\right).$$

Solving for the ideal staff reading gives

$$L_{\mathrm{ideal}} = \frac{F}{2\tan(\theta/2)}.$$

This notebook compares the observed staff reading with that idealized prediction.

## 3. Import Packages

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams['figure.figsize'] = (8, 5)

## 4. Load the Canonical Survey

In [ ]:
# Find the repository root by walking upward until data/ is found.
repo_root = Path.cwd().resolve()
while repo_root != repo_root.parent and not (repo_root / 'data').exists():
    repo_root = repo_root.parent

survey_dir = repo_root / 'data' / 'examples' / 'SUR-CAL-2026-001'

with open(survey_dir / 'survey.json', encoding='utf-8') as f:
    survey = json.load(f)

observations = pd.read_csv(survey_dir / 'observations.csv')
notes = pd.read_csv(survey_dir / 'notes.csv')

print(f"Loaded {len(observations)} observations from {survey['survey_id']}")

## 5. Prepare Analysis-Friendly Columns

Short aliases are created inside the notebook. The canonical CSV remains unchanged.

In [ ]:
cal = observations.copy()

aliases = {
    'calibration.target_distance': 'target_distance',
    'calibration.target_width': 'target_width',
    'calibration.target_id': 'target_id',
}

for source, alias in aliases.items():
    if source in cal.columns:
        cal[alias] = cal[source]

required = ['observation_id', 'fiducial_id', 'staff_reading', 'target_distance', 'target_width']
missing = [column for column in required if column not in cal.columns]
if missing:
    raise KeyError(f'Missing required analysis columns: {missing}')

cal.head()

## 6. Confirm Units

The equations require `target_width`, `target_distance`, `fiducial_id`, and `staff_reading` to use compatible length units.

Campaign 001 appears to use inches for the calibration geometry and staff readings. Confirm this against the campaign documentation before interpreting the results.

In [ ]:
cal[['target_width', 'target_distance', 'fiducial_id', 'staff_reading']].describe()

## 7. Calculate Target Angular Width

In [ ]:
cal['target_angle_rad'] = 2 * np.arctan(
    cal['target_width'] / (2 * cal['target_distance'])
)
cal['target_angle_deg'] = np.degrees(cal['target_angle_rad'])

cal[
    ['observation_id', 'target_width', 'target_distance', 'target_angle_deg']
].head()

## 8. Calculate Ideal Staff Reading

In [ ]:
cal['ideal_staff_reading'] = cal['fiducial_id'] / (
    2 * np.tan(cal['target_angle_rad'] / 2)
)

cal['staff_reading_residual'] = (
    cal['staff_reading'] - cal['ideal_staff_reading']
)

cal[
    [
        'observation_id',
        'fiducial_id',
        'staff_reading',
        'ideal_staff_reading',
        'staff_reading_residual',
    ]
].head()

A positive residual means the observed crosspiece position was farther from the eye than the ideal geometric prediction. A negative residual means it was closer.

## 9. Observed vs Ideal Staff Reading

In [ ]:
plt.scatter(cal['ideal_staff_reading'], cal['staff_reading'])

limits = [
    min(cal['ideal_staff_reading'].min(), cal['staff_reading'].min()),
    max(cal['ideal_staff_reading'].max(), cal['staff_reading'].max()),
]
plt.plot(limits, limits, linestyle='--')

plt.xlabel('Ideal Staff Reading')
plt.ylabel('Observed Staff Reading')
plt.title('Observed vs Ideal Cross-Staff Reading')
plt.tight_layout()
plt.show()

Points on the dashed line agree with the ideal geometry. Systematic displacement from the line may indicate bias in the observer-instrument system.

## 10. Residual Distribution

In [ ]:
cal['staff_reading_residual'].hist(bins=12)
plt.axvline(0, linestyle='--')
plt.xlabel('Observed minus Ideal Staff Reading')
plt.ylabel('Count')
plt.title('Distribution of Geometry-Model Residuals')
plt.tight_layout()
plt.show()

## 11. Residuals by Ideal Staff Reading

In [ ]:
for fiducial, subset in cal.groupby('fiducial_id'):
    plt.scatter(
        subset['ideal_staff_reading'],
        subset['staff_reading_residual'],
        label=fiducial,
    )

plt.axhline(0, linestyle='--')
plt.xlabel('Ideal Staff Reading')
plt.ylabel('Residual')
plt.title('Residuals Across the Instrument Range')
plt.legend(title='Fiducial')
plt.tight_layout()
plt.show()

Look for curvature, increasing spread, fiducial-specific offsets, or concentrations of large residuals near the practical limits of the instrument.

## 12. Residual Summary

In [ ]:
residual_summary = cal['staff_reading_residual'].agg(
    ['count', 'mean', 'median', 'std', 'min', 'max']
)

residual_summary

In [ ]:
residuals_by_fiducial = (
    cal.groupby('fiducial_id')['staff_reading_residual']
    .agg(['count', 'mean', 'median', 'std', 'min', 'max'])
)

residuals_by_fiducial

## 13. Inspect the Largest Absolute Residuals

In [ ]:
largest_residuals = (
    cal.assign(
        absolute_residual=cal['staff_reading_residual'].abs()
    )
    .sort_values('absolute_residual', ascending=False)
)

largest_residuals[
    [
        'observation_id',
        'fiducial_id',
        'target_width',
        'target_distance',
        'staff_reading',
        'ideal_staff_reading',
        'staff_reading_residual',
        'notes_id',
    ]
].head(10)

Large residuals should not be removed automatically. Review the underlying configuration, acquisition order, and observation notes before deciding how to interpret them.

## 14. Revisit the Edge-of-Capability Configuration

In [ ]:
edge_subset = cal[
    (cal['fiducial_id'] == 0.25)
    & (cal['target_width'] == 0.5)
].copy()

edge_subset[
    [
        'observation_id',
        'target_distance',
        'staff_reading',
        'ideal_staff_reading',
        'staff_reading_residual',
        'notes_id',
    ]
]

In [ ]:
plt.scatter(
    edge_subset['ideal_staff_reading'],
    edge_subset['staff_reading'],
)
for _, row in edge_subset.iterrows():
    plt.annotate(
        row['observation_id'].split('-')[-1],
        (row['ideal_staff_reading'], row['staff_reading']),
    )

limits = [
    min(edge_subset['ideal_staff_reading'].min(), edge_subset['staff_reading'].min()),
    max(edge_subset['ideal_staff_reading'].max(), edge_subset['staff_reading'].max()),
]
plt.plot(limits, limits, linestyle='--')
plt.xlabel('Ideal Staff Reading')
plt.ylabel('Observed Staff Reading')
plt.title('0.25 Fiducial with 0.5 Target Width')
plt.tight_layout()
plt.show()

## 15. Save Derived Geometry Table

In [ ]:
output_dir = repo_root / 'analysis' / 'tables' / 'cross-staff-calibration-001'
output_dir.mkdir(parents=True, exist_ok=True)

geometry_columns = [
    'observation_id',
    'fiducial_id',
    'target_width',
    'target_distance',
    'staff_reading',
    'target_angle_rad',
    'target_angle_deg',
    'ideal_staff_reading',
    'staff_reading_residual',
    'notes_id',
]

cal[geometry_columns].to_csv(
    output_dir / 'geometry-model-observations.csv',
    index=False,
)

residuals_by_fiducial.to_csv(
    output_dir / 'geometry-model-residuals-by-fiducial.csv'
)

print(f'Saved geometry-model tables to: {output_dir}')

## 16. Interpretation Notes

The overall relationship between the observed staff readings and the ideal geometric model is encouraging. Most observations follow the expected geometric trend closely, suggesting that the simple cross-staff model captures the dominant behavior of the instrument.

The largest deviations occur near the apparent operational limits of the instrument. In particular, the 0.5-inch calibration target was difficult to resolve at a distance of 4 feet and could not be measured at the standard 10-foot calibration distance. These observations are retained because they represent legitimate measurements collected near the practical resolution limit of the instrument rather than obvious recording errors.

One observation using the 4-inch fiducial exhibits a comparatively large residual. At present there is insufficient evidence to determine whether this represents an isolated measurement error, an issue specific to that fiducial, or normal experimental variability. This configuration should be revisited in a future calibration campaign.

No clear evidence of a systematic offset is apparent. Likewise, the residual plots do not suggest an obvious increase in variability across the operating range, although the present experimental design may not provide sufficient power to detect subtle trends.

Subjectively, observations became more difficult as the crosspiece approached the eye because maintaining focus became increasingly challenging. Although this effect was noticeable during data collection, it is not immediately apparent in the residual plots, particularly in the "Residuals Across the Instrument Range" figure. Additional calibration measurements would be required to determine whether this perceived limitation produces a measurable systematic effect.

Overall, these results suggest that the cross-staff performs well over most of its intended operating range while also illustrating the importance of characterizing the practical limits of an observing instrument.

## 17. Next Step

If the ideal geometric relationship captures the broad pattern but leaves systematic residual structure, the next analysis can estimate an empirical calibration relationship and compare it with the ideal model.